# Lineage y Memory que cambian decisiones · 2.1.4

**Input → proceso → output:** el pago PAY-001 llega con 5000 centavos MXN;
un Agent calcula una comisión de 50 y otro verifica su evidencia y decide.
El primer run aprueba. El segundo recibe exactamente el mismo input y lo manda
a revisión por duplicado gracias al episodio conservado en Memory.

Todo es Python determinista, coordinado por LangGraph: sin modelos, claves ni
inferencia. Requiere el SDK local **2.1.4** y sus dependencias; no se instala
nada desde el notebook. Ejecuta las celdas en orden desde un kernel limpio.

| Canal | Contenido | Papel en la decisión |
|---|---|---|
| State | payment_id, amount, fee, decision | Datos del pago actual |
| Context.data / sources | Política y su referencia | Comisión y límite |
| Context.lineage | Evidencia del Agent precedente | Exigir y verificar el cálculo |
| Context.episodes | Ejecución anterior con linaje compacto | Detectar aprobación anterior del mismo pago |

Las capturas de Context se guardan aparte para inspección; nunca se devuelven
como estado del Graph. Su linaje es evidencia observada, no pensamiento privado.

In [2]:
import pydantic
import agentic_systems as toolkit

assert toolkit.__version__ == "2.1.4"
print("SDK:", toolkit.__version__)

SDK: 2.1.4.dev8


## 1. Estado pequeño y Tools explícitas

`strict=True` rechaza coerciones de tipos y `extra="forbid"` rechaza campos
inesperados. Los campos opcionales son resultados todavía no calculados.
El revisor no acepta una comisión solo porque esté en el estado: necesita el
linaje exitoso de fees y exige consistencia con la política actual.
Después consulta evidencia estructurada de aprobaciones anteriores en Memory.

In [3]:
class Payment(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(strict=True, extra="forbid")
    payment_id: str = pydantic.Field(min_length=1)
    amount: int = pydantic.Field(gt=0)
    fee: int | None = None
    decision: str | None = None

In [4]:
# Capturas de observación: no forman parte del estado.
received_contexts = []

@toolkit.tool
def calculate_fee(amount: int, *, context: toolkit.Context) -> dict:
    received_contexts.append({"agent": "fees", "context": context.to_dict()})
    return {"fee": amount * context.data["fee_bps"] // 10000}


@toolkit.tool
def decide(payment_id: str, amount: int, fee: int, *, context: toolkit.Context) -> dict:
    received_contexts.append({"agent": "reviewer", "context": context.to_dict()})

    # Current evidence must support the fee before this agent may approve it.
    fee_evidence = [row for row in context.lineage if row["name"] == "fees"]
    if not fee_evidence or not fee_evidence[-1]["ok"]:
        raise ValueError("Missing successful fees lineage")
    expected_fee = amount * context.data["fee_bps"] // 10000
    if fee_evidence[-1]["data"].get("fee") != fee or fee != expected_fee:
        raise ValueError("Fee is inconsistent with lineage or current policy")

    # Previous approval of the same business identifier changes the decision.
    # Use structured recorded evidence, never parse a human-readable explanation.
    for episode in context.episodes:
        if not episode["ok"]:
            continue
        for step in episode["lineage"]["steps"]:
            if step["kind"] != "execution" or step["title"] != "payments":
                continue
            evidence = step["evidence"]
            previous = evidence.get("data", {})
            if (evidence["ok"] and previous.get("payment_id") == payment_id
                    and previous.get("decision") == "approve"):
                return {"decision": "review_duplicate"}

    return {
        "decision": "approve" if amount + fee <= context.data["ceiling"] else "review"
    }

## 2. Un Graph por Agent y un Graph para el System

fees organiza el cálculo; reviewer organiza la decisión. El System los conecta
en ese orden. No hay lectura manual de una traza global ni historial añadido
al modelo Payment: el contexto se entrega al ejecutar cada Tool.

In [5]:
fee_graph =toolkit.Graph(Payment)
fee_graph.add_node("calculate_fee", calculate_fee)
fee_graph.add_edge(toolkit.START, "calculate_fee")
fee_graph.add_edge("calculate_fee", toolkit.END)
fee_agent = toolkit.Agent("fees", fee_graph)

review_graph = toolkit.Graph(Payment)
review_graph.add_node("decide", decide)
review_graph.add_edge(toolkit.START, "decide")
review_graph.add_edge("decide", toolkit.END)
reviewer = toolkit.Agent("reviewer", review_graph)

payment_graph = toolkit.Graph(Payment)
payment_graph.add_node("fees", fee_agent)
payment_graph.add_node("reviewer", reviewer)
payment_graph.add_edge(toolkit.START, "fees")
payment_graph.add_edge("fees", "reviewer")
payment_graph.add_edge("reviewer", toolkit.END)

## 3. Política y Memory compartida desde el System

Memory conserva hasta dos episodios locales. En cada entrada se obtiene una vista
de los episodios previos que heredan los Agents. Al terminar, el System recuerda
su resultado. `compile()` prepara el harness sin ejecutar Tools ni escribir memoria.

In [6]:
memory =toolkit.Memory(capacity=2)
system = toolkit.System(
    "payments",
    payment_graph,
    context=toolkit.Context(
        {"currency": "MXN", "fee_bps": 100, "ceiling": 10000},
        sources=("payment-policy-v1",),
    ),
    memory=memory,
)

harness = system.compile()
assert received_contexts == [] and memory.read() == []

## 4. Primera ejecución: hay evidencia actual, todavía no hay historia

fees recibe lineage y episodes vacíos. reviewer recibe la evidencia actual de
fees, pero ningún episodio anterior: PAY-001 puede aprobarse.

In [7]:
payment = {"payment_id": "PAY-001", "amount": 5000}
first = harness.run(payment)
assert first.ok
assert first.data == {**payment, "fee": 50, "decision": "approve"}
first_contexts = list(received_contexts)
assert len(first_contexts) == 2
assert len(memory.read()) == 1
print(first.human_results())

payments: COMPLETED
Execution: run-10043c15b7ac4ed3bd549d001b112e61
Input: {"payment_id":"PAY-001","amount":5000}
Output: {"payment_id":"PAY-001","amount":5000,"fee":50,"decision":"approve"}
Process:
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}


### Context exacto recibido en la primera ejecución

Se muestra el objeto completo: política, fuentes, linaje actual y episodios.

In [8]:
import json

for row in first_contexts:
    print("Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [],
  "lineage": []
}
Agent: reviewer
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [],
  "lineage": [
    {
      "data": {
        "fee": 50
      },
      "execution_id": "run-46b56f4ed84e4d88b774996a8aeeb6db",
      "name": "fees",
      "ok": true
    }
  ]
}


## 5. Segunda ejecución: el mismo input cambia de decisión por Memory

Se reutiliza el harness y su Memory. La comisión y el límite siguen iguales.
reviewer encuentra en el linaje del episodio anterior que PAY-001 ya fue aprobado
y devuelve review_duplicate. La ejecución termina correctamente: pedir revisión
es una decisión de negocio, no un fallo del runtime.

In [9]:
second = harness.run(payment)
assert second.ok
assert second.data == {**payment, "fee": 50, "decision": "review_duplicate"}
second_contexts = list(received_contexts[2:])
assert len(second_contexts) == 2
assert len(memory.read()) == 2
print(second.human_results())

payments: COMPLETED
Execution: run-3bfadcdc65084eba92fb8075ac993539
Input: {"payment_id":"PAY-001","amount":5000}
Output: {"payment_id":"PAY-001","amount":5000,"fee":50,"decision":"review_duplicate"}
Process:
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"review_duplicate"}


### Context exacto recibido en la segunda ejecución

Ambos reciben el episodio de first en episodes. Solo reviewer recibe además la
observación actual de fees en lineage. El primer agente no confunde historia de
la ejecución anterior con trabajo ya realizado en esta ejecución.

In [10]:
for row in second_contexts:
    print("Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [
    {
      "context": {
        "episode_ids": [],
        "fingerprint": "2b78cb800cc7bca055249311d20ea1121ffd1004657a1ef7a0c2c58c7d8ddaad",
        "memory_revision": 0,
        "sources": [
          "payment-policy-v1"
        ]
      },
      "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
      "lineage": {
        "answer": "{\"payment_id\": \"PAY-001\", \"amount\": 5000, \"fee\": 50, \"decision\": \"approve\"}",
        "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
        "failed_steps": 0,
        "ok": true,
        "omitted_steps": 0,
        "steps": [
          {
            "evidence": {
              "data": {
                "amount": 5000,
                "decision": "approve",
                "fee": 50,
                "payment_id": "PAY-001"
              },
              "errors": [],
         

## 6. Controles que demuestran la dependencia causal

1. Mismo input y mismo Graph sin Memory: vuelve a approve.
2. fee=50 en el estado, sin linaje previo de fees: la ejecución falla.
3. fees registra fee=51: el linaje existe, pero contradice la política; falla.

Los controles usan Systems independientes y no escriben la Memory del ejemplo.

In [11]:
without_memory = toolkit.System(
    "payments", payment_graph, context=system.context,
).run(payment)
assert without_memory.ok and without_memory.data["decision"] == "approve"
print("Sin Memory:")
print(without_memory.human_results())

missing_lineage = toolkit.System(
    "missing_evidence", review_graph, context=system.context,
).run({**payment, "fee": 50})
assert not missing_lineage.ok
assert "Missing successful fees lineage" in str(missing_lineage.errors)
print("\nSin linaje previo:")
print(missing_lineage.human_results())

Sin Memory:
payments: COMPLETED
Execution: run-b80204c715154a9bb7afbe5814bae4f4
Input: {"payment_id":"PAY-001","amount":5000}
Output: {"payment_id":"PAY-001","amount":5000,"fee":50,"decision":"approve"}
Process:
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}

Sin linaje previo:
missing_evidence: FAILED
Execution: run-e7218c417e1e4a6c8257db8a63caf58b
Input: {"payment_id":"PAY-001","amount":5000,"fee":50}
Output: {}
Errors: [{"code":"_NodeFailure","message":"9 validation errors for Payment\ncategory\n  Extra inputs are not permitted [type=extra_forbidden, input_value='provider', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden\nmessage\n  Extra inputs are not permitted [type=extra_forbidden, input_value='Missing successful fees lineage', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidd

In [12]:
def incorrect_fee(state):
    return {"fee": 51}


bad_graph = toolkit.Graph(Payment)
bad_graph.add_node("fees", incorrect_fee)
bad_graph.add_node("reviewer", reviewer)
bad_graph.add_edge(toolkit.START, "fees")
bad_graph.add_edge("fees", "reviewer")
bad_graph.add_edge("reviewer", toolkit.END)

inconsistent_lineage = toolkit.System(
    "inconsistent_evidence", bad_graph, context=system.context,
).run(payment)
assert not inconsistent_lineage.ok
assert "Fee is inconsistent" in str(inconsistent_lineage.errors)
assert len(memory.read()) == 2
print(inconsistent_lineage.human_results())

inconsistent_evidence: FAILED
Execution: run-3d5bdf410af246b688cbdbb820c9757b
Input: {"payment_id":"PAY-001","amount":5000}
Output: {}
Errors: [{"code":"_NodeFailure","message":"Required node failed: reviewer"},{"code":"_NodeFailure","message":"9 validation errors for Payment\ncategory\n  Extra inputs are not permitted [type=extra_forbidden, input_value='provider', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden\nmessage\n  Extra inputs are not permitted [type=extra_forbidden, input_value='Fee is inconsistent with...neage or current policy', input_type=str]\n    For further informa…
Process:
- Step fees: completed
- Agent reviewer: failed
- Tool decide: failed -> {"category":"provider","message":"Fee is inconsistent with lineage or current policy","provider":"python-runtime","framework":"native","code":"ValueError","retryable":false,"cause_type":"ValueError","details":{},"error_type":"ValueError"}


## 7. Resumen verificado: cuatro Context, cuatro campos de negocio

In [13]:
main_contexts = first_contexts + second_contexts
assert len(main_contexts) == 4
print("Run | Agent    | lineage actual | episodios previos")
for index, row in enumerate(main_contexts):
    context = row["context"]
    assert context["data"] == {"currency": "MXN", "fee_bps": 100, "ceiling": 10000}
    assert context["sources"] == ["payment-policy-v1"]
    assert len(context["episodes"]) == index // 2
    if row["agent"] == "fees":
        assert context["lineage"] == []
    else:
        assert any(item["name"] == "fees" and item["ok"] for item in context["lineage"])
    if index >= 2:
        assert context["episodes"][0]["execution_id"] == first.execution_id
    print(index // 2 + 1, row["agent"], len(context["lineage"]), len(context["episodes"]), sep=" | ")

assert set(first.data) == set(second.data) == {"payment_id", "amount", "fee", "decision"}
print("\nPrimer resultado:", first.data)
print("Segundo resultado:", second.data)
print("Campos de estado:", list(Payment.model_fields))
print("Episodios conservados:", len(memory.read()))
print("Dos ejecuciones y tres controles: correctos.")

Run | Agent    | lineage actual | episodios previos
1 | fees | 0 | 0
1 | reviewer | 1 | 0
2 | fees | 0 | 1
2 | reviewer | 1 | 1

Primer resultado: {'payment_id': 'PAY-001', 'amount': 5000, 'fee': 50, 'decision': 'approve'}
Segundo resultado: {'payment_id': 'PAY-001', 'amount': 5000, 'fee': 50, 'decision': 'review_duplicate'}
Campos de estado: ['payment_id', 'amount', 'fee', 'decision']
Episodios conservados: 2
Dos ejecuciones y tres controles: correctos.


## 8. Environment: Memory evoluciona entre episodios

Procesamos PAY-002 dos veces con el mismo System. observe entrega solo los campos
del pago; expected_decision queda fuera del input del System y sirve como oráculo
del Environment. update registra una recompensa explícita y captura Memory aparte.
La capacidad es dos: el número de episodios ya no crece, pero cambian sus IDs.
No adjuntamos otra Memory al Environment: queremos observar la del System.

In [14]:
environment_memory_history = []


def observe_payment(record, state):
    return {"payment_id": record["payment_id"], "amount": record["amount"]}


def update_episode(state, record, result):
    environment_memory_history.append({
        "payment_id": record["payment_id"],
        "decision": result.data["decision"],
        "memory": memory.read(),
    })
    reward = int(result.ok and result.data["decision"] == record["expected_decision"])
    return {"processed": state["processed"] + 1,
            "reward": reward, "total_reward": state["total_reward"] + reward}


episode_graph = toolkit.Graph(Payment)
episode_graph.add_node("payment", system)
episode_graph.add_edge(toolkit.START, "payment")
episode_graph.add_edge("payment", toolkit.END)

environment = toolkit.Environment(
    "payment_day", graph=episode_graph,
    records=(
        {"payment_id": "PAY-002", "amount": 5000, "expected_decision": "approve"},
        {"payment_id": "PAY-002", "amount": 5000, "expected_decision": "review_duplicate"},
    ),
    state={"processed": 0, "reward": 0, "total_reward": 0},
    observe=observe_payment, update=update_episode,
)

In [15]:
memory_before_environment = memory.read()
environment_context_start = len(received_contexts)
episode = environment.run(timeout_s=30)
assert episode.ok
assert [row["decision"] for row in episode.data["outputs"]] == ["approve", "review_duplicate"]
assert episode.data["summary"]["state"]["total_reward"] == 2
environment_contexts = list(received_contexts[environment_context_start:])
assert len(environment_contexts) == 4 and len(environment_memory_history) == 2
assert all(len(row["memory"]) == 2 for row in environment_memory_history)
assert environment_memory_history[0]["memory"][0]["execution_id"] == second.execution_id
assert environment_memory_history[1]["memory"][0]["execution_id"] == environment_memory_history[0]["memory"][1]["execution_id"]
assert [e["execution_id"] for e in environment_contexts[2]["context"]["episodes"]] == [e["execution_id"] for e in environment_memory_history[0]["memory"]]
print(episode.human_results())
print("\nEstado del Environment:", episode.data["summary"]["state"])
print("Antes:", [e["execution_id"] for e in memory_before_environment])
for index, row in enumerate(environment_memory_history, 1):
    print("Después del episodio", index, row["decision"], [e["execution_id"] for e in row["memory"]])

payment_day: COMPLETED
Execution: run-caa00e0704bb46949cc7ab123bcb1b19
Input: {"records_observed":[{"payment_id":"PAY-002","amount":5000,"expected_decision":"approve"},{"payment_id":"PAY-002","amount":5000,"expected_decision":"review_duplicate"}]}
Output: {"outputs":[{"payment_id":"PAY-002","amount":5000,"fee":50,"decision":"approve"},{"payment_id":"PAY-002","amount":5000,"fee":50,"decision":"review_duplicate"}],"summary":{"state":{"processed":2,"reward":1,"total_reward":2},"processed":2,"terminated":true,"truncated":false,"reason":"records_exhausted"}}
Process:
- System payments: completed
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}
- System payments: completed
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
2 operations omitted; inspect lineage for full evidence.

Estado del Environment: {'processed': 2, 'reward': 1, 'total_reward': 2}
Antes: ['run-10043c15b7

### Context exactos durante Environment

La segunda pareja de Tools recibe el episodio del primer pago PAY-002. La lista de observación no se añade al estado.

In [16]:
for index, row in enumerate(environment_contexts):
    print("Episodio", index // 2 + 1, "Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

Episodio 1 Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [
    {
      "context": {
        "episode_ids": [],
        "fingerprint": "2b78cb800cc7bca055249311d20ea1121ffd1004657a1ef7a0c2c58c7d8ddaad",
        "memory_revision": 0,
        "sources": [
          "payment-policy-v1"
        ]
      },
      "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
      "lineage": {
        "answer": "{\"payment_id\": \"PAY-001\", \"amount\": 5000, \"fee\": 50, \"decision\": \"approve\"}",
        "execution_id": "run-10043c15b7ac4ed3bd549d001b112e61",
        "failed_steps": 0,
        "ok": true,
        "omitted_steps": 0,
        "steps": [
          {
            "evidence": {
              "data": {
                "amount": 5000,
                "decision": "approve",
                "fee": 50,
                "payment_id": "PAY-001"
              },
              "errors": []

## 9. Eval: episodios vivos dentro del caso, Memory aislada entre casos

Evaluamos el mismo Environment dos veces, cada caso con PAY-003 repetido.
Cada caso parte de la misma Memory de producción: PAY-002. Dentro de cada caso,
la primera aprobación de PAY-003 alimenta su segundo episodio y cambia la decisión.
El siguiente caso vuelve a la base de producción: debe aprobar su primer PAY-003.

El check lee la Memory capturada por cierre: también recibe la vista privada del
caso. Al salir de Eval, exigimos igualdad exacta de la Memory de producción.
Las listas de auditoría sí acumulan observaciones; Eval no aísla archivos,
variables arbitrarias, clientes SDK ni efectos externos.

In [17]:
eval_memory_views = []


def check_episode(result, case):
    decisions = [row["decision"] for row in result.data["outputs"]]
    eval_memory_views.append({"case": case["name"], "memory": memory.read()})
    return {"ok": decisions == ["approve", "review_duplicate"]
            and result.data["summary"]["state"]["total_reward"] == 2}


eval_records = (
    {"payment_id": "PAY-003", "amount": 5000, "expected_decision": "approve"},
    {"payment_id": "PAY-003", "amount": 5000, "expected_decision": "review_duplicate"},
)
evaluation = toolkit.Eval(
    "isolated_payment_days", unit=environment,
    cases=(
        {"name": "case-a", "input": {"records": eval_records}, "expected": {"ok": True}},
        {"name": "case-b", "input": {"records": eval_records}, "expected": {"ok": True}},
    ),
    checks=(check_episode,),
)

In [18]:
production_before_eval = memory.read()
production_ids = [e["execution_id"] for e in production_before_eval]
eval_context_start = len(received_contexts)
evaluated = evaluation.run(timeout_s=30)
eval_contexts = list(received_contexts[eval_context_start:])
assert evaluated.ok and evaluated.data["passed"] == 2
assert evaluated.meta["memory_isolation"] == "per_case"
assert memory.read() == production_before_eval
assert len(eval_contexts) == 8 and len(eval_memory_views) == 2
for offset in (0, 4):
    first_episode = eval_contexts[offset]["context"]["episodes"]
    second_episode = eval_contexts[offset + 2]["context"]["episodes"]
    assert [e["execution_id"] for e in first_episode] == production_ids
    assert [e["execution_id"] for e in second_episode] != production_ids
assert [e["execution_id"] for e in eval_memory_views[0]["memory"]] != [e["execution_id"] for e in eval_memory_views[1]["memory"]]
print(evaluated.human_results())
print("\nMemory producción antes:", production_ids)
for row in eval_memory_views:
    print("Vista privada", row["case"], [e["execution_id"] for e in row["memory"]])
print("Memory producción después:", [e["execution_id"] for e in memory.read()])
print("Igualdad exacta antes/después:", memory.read() == production_before_eval)

isolated_payment_days: COMPLETED
Execution: run-ee73b0dc43404921833d2506062bf4de
Input: {"cases_executed":[{"records":[{"payment_id":"PAY-003","amount":5000,"expected_decision":"approve"},{"payment_id":"PAY-003","amount":5000,"expected_decision":"review_duplicate"}]},{"records":[{"payment_id":"PAY-003","amount":5000,"expected_decision":"approve"},{"payment_id":"PAY-003","amount":5000,"expected_decision":"review_duplicate"}]}]}
Output: {"total":2,"passed":2,"failed":0}
Process:
- Environment payment_day: completed
- System payments: completed
- Agent fees: completed
- Tool calculate_fee: completed -> {"fee":50}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}
- System payments: completed
- Agent fees: completed
14 operations omitted; inspect lineage for full evidence.

Memory producción antes: ['run-9956139e296e4b17ab860471a1729c2c', 'run-bc6f52d4511545d3b632dc4f14753b83']
Vista privada case-a ['run-65acfe1e2794468a8ae0c9b3193c63d1', 'run-8044b9c498b74ae187

### Context exactos durante Eval

Se muestran los ocho Context: dos casos × dos episodios × dos Agents. Los dos casos arrancan con los mismos IDs de producción; las escrituras del primero no llegan al segundo.

In [19]:
for index, row in enumerate(eval_contexts):
    print("Caso", index // 4 + 1, "episodio", (index % 4) // 2 + 1, "Agent:", row["agent"])
    print(json.dumps(row["context"], ensure_ascii=False, indent=2))

for result in (first, second, episode, evaluated):
    assert result.check_invariants().ok
    assert result.usage.get("inference_requests", 0) == 0
assert set(Payment.model_fields) == {"payment_id", "amount", "fee", "decision"}
print("Environment y Eval correctos; Memory de producción intacta tras Eval.")

Caso 1 episodio 1 Agent: fees
{
  "data": {
    "ceiling": 10000,
    "currency": "MXN",
    "fee_bps": 100
  },
  "sources": [
    "payment-policy-v1"
  ],
  "episodes": [
    {
      "context": {
        "episode_ids": [
          "run-10043c15b7ac4ed3bd549d001b112e61",
          "run-3bfadcdc65084eba92fb8075ac993539"
        ],
        "fingerprint": "455e5946dbc0274969ce803033c3858d97d35144b265b6675c62bc42f8c551fa",
        "memory_revision": 2,
        "sources": [
          "payment-policy-v1"
        ]
      },
      "execution_id": "run-9956139e296e4b17ab860471a1729c2c",
      "lineage": {
        "answer": "{\"payment_id\": \"PAY-002\", \"amount\": 5000, \"fee\": 50, \"decision\": \"approve\"}",
        "execution_id": "run-9956139e296e4b17ab860471a1729c2c",
        "failed_steps": 0,
        "ok": true,
        "omitted_steps": 0,
        "steps": [
          {
            "evidence": {
              "data": {
                "amount": 5000,
                "decision": "appro

## Alcance y límites

El linaje de un Agent o System puede aportar contexto a otro. Context también
puede incluir política y fuentes externas: mostrar solo lineage es una proyección,
no el Context completo. Aquí inspeccionamos todo, pero las decisiones consultan
campos estructurados; no analizan el texto de human_results.

Memory es local, acotada y contiene resúmenes. La ausencia de una aprobación en
esos episodios **no prueba** que nunca ocurrió. Esta demostración no sustituye
un registro persistente de idempotencia ni evita duplicados concurrentes.
Los cuatro campos de negocio no contienen Context, episodios ni árboles de
RunResult; internamente el grafo usa referencias acotadas a evidencia.

La validación cubre este escenario determinista y sus controles. No certifica
proveedores externos ni nuevas versiones. No se publica ningún artefacto.

[Programa base de dos Agents](example.py); Environment y Eval se declaran directamente en este notebook. [Explicación y evidencia](README.md).